# 指标口径对齐

> 对应博客篇目：《[性能剖析（02）：推理服务 profiling](https://lrypcy.github.io/2026/09/29/profiling-02-inference/)》
>
> 代码逐字摘自博客仓库 `tools/profiling_bench/serving_schema.py`（[lrypcy/ipynbs](https://github.com/lrypcy/ipynbs)）。

## 为什么需要这一步

「TTFT」「吞吐」「goodput」这些词在工具之间**同名不同义**。不逐字段钉一遍，就会拿 A 工具的字段去比 B 工具的字段，然后得出一个自己都不知道在比什么的结论。

本 notebook 做三件事：

1. **字段映射表**：每个能对齐的字段标明来源与可信度（已核验 / 条件出现 / 版本相关 / 推断 / 自造）。
2. **能力边界表**：明确列出**队列模型给不出的量**。报得出来的和报不出来的都要写清楚，否则读者会以为模型能替代压测。
3. **ITL 与 TPOT 的分母演示**：这两个量在流式输出下不是一回事，差多少有一个闭式表达，**不依赖任何仿真**。

## 字段名的可信度分级（不是所有名字都同样硬）

| 标签 | 含义 |
|:---|:---|
| 已核验 | vLLM 文档或 `benchmark_serving.py` 源码里直接出现的名字 |
| 条件出现 | 只有 `--metric-percentiles` 里包含该分位才有；**默认值是 99**，所以 `p90_*` 默认跑出来的结果文件里没有这个键 |
| 版本相关 | 来自结果文件的实际用法，不同版本可能增删；跨版本对比前先 `--save-result` 跑一次看实际键名 |
| 推断 | 由文档描述推出，未经一手确认。**不要拿推断项做自动化断言** |
| 自造 | 本系列自己的合成量，工具里没有对应字段 |

## 第 0 步 · 环境与依赖

这个 notebook 的第 3 节要真跑一次队列模型，所以先把 `serving_model` 里用到的符号摊开（与 [profiling-02-serving-model](../profiling-02-serving-model/) 同一份代码）。

In [1]:
import argparse
import json
import sys
import argparse
import json
import math
import random
import sys
try:  # numpy 只用于采样与分位，缺了也能跑（stdlib 回退）
    import numpy as _np
except ImportError:  # pragma: no cover
    _np = None

In [2]:
# ---- 摘自 tools/profiling_bench/serving_model.py ----
DEFAULT_COST = dict(
    prefill_floor_ms=3.0,      # prefill 的与批量无关部分
    prefill_ms_per_tok=0.080,  # 峰值约 12500 prefill tok/s
    decode_floor_ms=12.0,      # decode 单步下限（权重搬运主导）
    decode_ms_per_seq=0.25,    # 膝盖在 48 路，吞吐上限约 4000 tok/s
)



class ServerCfg(object):
    """一次压测的服务器侧配置。字段名刻意贴 vLLM 的启动参数。"""

    def __init__(self,
                 max_num_seqs=32,
                 max_num_batched_tokens=2048,
                 kv_token_capacity=262144,
                 block_size=16,
                 preempt=True,
                 **cost):
        self.max_num_seqs = max_num_seqs
        self.max_num_batched_tokens = max_num_batched_tokens
        self.kv_token_capacity = kv_token_capacity
        self.block_size = block_size
        self.preempt = preempt
        c = dict(DEFAULT_COST)
        c.update(cost)
        self.prefill_floor_ms = c["prefill_floor_ms"]
        self.prefill_ms_per_tok = c["prefill_ms_per_tok"]
        self.decode_floor_ms = c["decode_floor_ms"]
        self.decode_ms_per_seq = c["decode_ms_per_seq"]

    # --- 单步代价：roofline 两段 ---
    def prefill_step_ms(self, n_tok):
        if n_tok <= 0:
            return 0.0
        return max(self.prefill_floor_ms, self.prefill_ms_per_tok * n_tok)

    def decode_step_ms(self, n_seq):
        if n_seq <= 0:
            return 0.0
        return max(self.decode_floor_ms, self.decode_ms_per_seq * n_seq)

    # --- 膝盖位置与饱和上限，都是解析值 ---
    @property
    def decode_knee(self):
        """decode 从「与批量无关」转为「随批量线性」的并发数。"""
        if self.decode_ms_per_seq <= 0:
            return float("inf")
        return self.decode_floor_ms / self.decode_ms_per_seq

    @property
    def decode_throughput_ceiling(self):
        """输出吞吐的渐近上限（tok/s）。"""
        if self.decode_ms_per_seq <= 0:
            return float("inf")
        return 1000.0 / self.decode_ms_per_seq

    def kv_blocks(self, tokens):
        return int(math.ceil(tokens / float(self.block_size)))

    def as_dict(self):
        return dict(
            max_num_seqs=self.max_num_seqs,
            max_num_batched_tokens=self.max_num_batched_tokens,
            kv_token_capacity=self.kv_token_capacity,
            block_size=self.block_size,
            preempt=self.preempt,
            prefill_floor_ms=self.prefill_floor_ms,
            prefill_ms_per_tok=self.prefill_ms_per_tok,
            decode_floor_ms=self.decode_floor_ms,
            decode_ms_per_seq=self.decode_ms_per_seq,
        )



class _Req(object):
    __slots__ = ("idx", "arrival_ms", "in_len", "out_len",
                 "in_done", "out_done", "chunk", "ttft_ms", "e2el_ms",
                 "preempted", "rejected", "finished")

    def __init__(self, idx, arrival_ms, in_len, out_len):
        self.idx = idx
        self.arrival_ms = arrival_ms
        self.in_len = in_len
        self.out_len = out_len
        self.in_done = 0
        self.out_done = 0
        self.chunk = 0
        self.ttft_ms = None
        self.e2el_ms = None
        self.preempted = False
        self.rejected = False
        self.finished = False

    def kv_tokens(self, block_size):
        used = self.in_len + max(self.out_done - 1, 0)
        return int(math.ceil(used / float(block_size))) * block_size



def arrival_times(pattern, rate, n, seed=0, shape=1.0):
    """生成到达时刻（秒）。返回 (times, cv_realized)。

    rate = 平均到达率 λ（req/s）。三种模式的**均值相同、分布不同**，
    这正是「三者不可互推」的来源：同一 λ 下队列行为可以差很多。

    - `constant`：等间隔，CV = 0。平滑到不真实，但它是复现性最好的基线。
    - `poisson` ：指数间隔，CV = 1。λ 就是指数分布的率参数。
    - `gamma`   ：Gamma(shape=k, rate=k·λ)，均值 = 1/λ，CV = 1/√k。
      k = 1 退化成 Poisson；k > 1 更规则；k < 1 更突发。

    注意一处容易踩的坑：**工具的旋钮名字和分布的平滑程度不一定同向。**
    不同实现里「burstiness」可能映射到 shape 本身，也可能映射到它的倒数。
    所以这里不按旋钮名推断，而是把**实测 CV 打出来**——
    换工具时先看 CV 对不对得上，再比数。

    CV 是实现值（有限样本），不是理论值：样本量小的时候会明显偏离 1/√k。
    """
    rng = random.Random(seed)
    if rate <= 0:
        raise ValueError("rate 必须为正")
    if n <= 0:
        return [], float("nan")

    gaps = []
    if pattern == "constant":
        gaps = [1.0 / rate] * n
    elif pattern == "poisson":
        for _ in range(n):
            u = rng.random()
            # 指数分布反变换，u 取 (0,1) 开区间
            gaps.append(-math.log(max(u, 1e-12)) / rate)
    elif pattern == "gamma":
        if shape <= 0:
            raise ValueError("shape 必须为正")
        for _ in range(n):
            gaps.append(_gamma_sample(rng, shape) / (shape * rate))
    else:
        raise ValueError("未知到达模式: %s" % pattern)

    times, acc = [], 0.0
    for g in gaps:
        acc += g
        times.append(acc)

    mean = sum(gaps) / len(gaps)
    if len(gaps) > 1 and mean > 0:
        var = sum((g - mean) ** 2 for g in gaps) / (len(gaps) - 1)
        cv = math.sqrt(var) / mean
    else:
        cv = 0.0
    return times, cv



def _gamma_sample(rng, shape):
    """Marsaglia-Tsang 法取 Gamma(shape, 1)。numpy 在就直接用。"""
    if _np is not None:
        return float(_np.random.default_rng(rng.randrange(2 ** 31)).gamma(shape))
    if shape < 1.0:  # 提升法：G(k) = G(k+1) · U^(1/k)
        u = max(rng.random(), 1e-12)
        return _gamma_sample(rng, shape + 1.0) * u ** (1.0 / shape)
    d = shape - 1.0 / 3.0
    c = 1.0 / math.sqrt(9.0 * d)
    while True:
        x = rng.gauss(0.0, 1.0)
        v = 1.0 + c * x
        if v <= 0:
            continue
        v = v ** 3
        u = max(rng.random(), 1e-12)
        if math.log(u) < 0.5 * x * x + d * (1.0 - v + math.log(v)):
            return d * v



def pct(values, q):
    """线性插值分位（与 numpy.percentile 的默认口径一致）。"""
    vals = sorted(v for v in values if v is not None)
    if not vals:
        return float("nan")
    if len(vals) == 1:
        return float(vals[0])
    pos = (len(vals) - 1) * q / 100.0
    lo = int(math.floor(pos))
    hi = int(math.ceil(pos))
    if lo == hi:
        return float(vals[lo])
    return float(vals[lo] + (vals[hi] - vals[lo]) * (pos - lo))



def meets_slo(rec, slo_ttft_ms, slo_tpot_ms):
    """判定单条请求是否达标。单 token 请求只有 TTFT 可言，不参与 TPOT 判定。"""
    if rec["ttft_ms"] is None:
        return False
    if slo_ttft_ms is not None and rec["ttft_ms"] > slo_ttft_ms:
        return False
    if rec["tpot_ms"] is None:
        return True
    if slo_tpot_ms is not None and rec["tpot_ms"] > slo_tpot_ms:
        return False
    return True



def summarize(records, meta, cfg, slo_ttft_ms=None, slo_tpot_ms=None):
    """把逐请求记录收成一份汇总。字段命名对齐 vLLM `--save-result`。"""
    dur_s = meta["duration_ms"] / 1000.0 if meta["duration_ms"] > 0 else float("nan")
    ttfts = [r["ttft_ms"] for r in records if r["ttft_ms"] is not None]
    tpots = [r["tpot_ms"] for r in records if r["tpot_ms"] is not None]
    e2els = [r["e2el_ms"] for r in records if r["e2el_ms"] is not None]
    # token 计数以**传入的 records** 为准，而不是 meta：
    # 传稳态子样本时，meta 里的计数属于整段，混用会把吞吐算高。
    # 时长仍用 meta 的整段挂钟——子样本切不出干净的分母，
    # 宁可让吞吐偏保守（分母偏大），也不要高估。
    n_out = sum(r["n_out"] for r in records) if records \
        else meta.get("total_output_tokens", 0)
    n_in = sum(r.get("in_len", 0) for r in records) if records \
        else meta.get("total_input_tokens", 0)

    out = dict(
        completed=len(records),
        duration=dur_s,
        total_input_tokens=n_in,
        total_output_tokens=n_out,
        request_throughput=(len(records) / dur_s) if dur_s and dur_s > 0 else float("nan"),
        output_throughput=(n_out / dur_s) if dur_s and dur_s > 0 else float("nan"),
        total_token_throughput=((n_in + n_out) / dur_s) if dur_s and dur_s > 0 else float("nan"),
        mean_ttft_ms=(sum(ttfts) / len(ttfts)) if ttfts else float("nan"),
        median_ttft_ms=pct(ttfts, 50),
        p90_ttft_ms=pct(ttfts, 90),
        p99_ttft_ms=pct(ttfts, 99),
        mean_tpot_ms=(sum(tpots) / len(tpots)) if tpots else float("nan"),
        median_tpot_ms=pct(tpots, 50),
        p90_tpot_ms=pct(tpots, 90),
        p99_tpot_ms=pct(tpots, 99),
        mean_e2el_ms=(sum(e2els) / len(e2els)) if e2els else float("nan"),
        median_e2el_ms=pct(e2els, 50),
        p99_e2el_ms=pct(e2els, 99),
        n_preemptions_in_model=meta["n_preempt"],
        n_rejected_in_model=meta["n_rejected"],
    )
    out["ttft_p99_over_p50"] = _ratio(out["p99_ttft_ms"], out["median_ttft_ms"])
    out["tpot_p99_over_p50"] = _ratio(out["p99_tpot_ms"], out["median_tpot_ms"])

    if slo_ttft_ms is not None or slo_tpot_ms is not None:
        ok = [r for r in records if meets_slo(r, slo_ttft_ms, slo_tpot_ms)]
        att = (len(ok) / len(records)) if records else float("nan")
        out["slo_ttft_ms"] = slo_ttft_ms
        out["slo_tpot_ms"] = slo_tpot_ms
        out["slo_attainment"] = att
        out["request_goodput"] = (len(ok) / dur_s) if dur_s and dur_s > 0 else float("nan")
        out["output_token_goodput"] = (
            (sum(r["n_out"] for r in ok) / dur_s) if dur_s and dur_s > 0 else float("nan")
        )
    return out



def cfg_in_len(records):
    return 0



def _pct_gain(a, b):
    """相对增益（百分数）。b 为 0 时返回 0，不抛除零。"""
    if not _finite(a) or not _finite(b) or b <= 0:
        return 0.0
    return (a / b - 1.0) * 100.0



def _ratio(a, b):
    if a is None or b is None or not _finite(a) or not _finite(b) or b <= 0:
        return float("nan")
    return a / b



def _finite(x):
    try:
        return math.isfinite(float(x))
    except (TypeError, ValueError):
        return False



def steady_state(records, drop):
    """丢掉启动瞬态，只留稳态样本。

    这不是可选项。closed loop 在 t=0 一次性放出全部并发，
    前 `并发数` 条请求要排队等 prefill 预算，尾部会被这段**爬坡**撑起来；
    把整段拿去算 p99，量到的是「冷启动有多慢」，不是「服务有多稳」。
    """
    ordered = sorted(records, key=lambda r: r["idx"])
    return ordered[drop:] if drop < len(ordered) else []



def steady_window(meta, subset):
    """给稳态子样本配一个**匹配的分母**。

    子样本的吞吐 = 子样本产出的 token / 子样本自身的时间窗，
    而不是 / 整段挂钟。用整段挂钟当分母会把吞吐系统性低估
    （丢得越多低估越狠）。窗口下界取子样本里最早的到达时刻；
    上界仍用整段结束时刻，边界上最多差一条请求。
    """
    if not subset:
        return dict(meta)
    t_lo = min(r["arrival_ms"] for r in subset)
    m = dict(meta)
    m["duration_ms"] = max(meta["duration_ms"] - t_lo, 1e-9)
    return m



def simulate(cfg, times_s, in_len, out_len, max_steps=2_000_000, closed=None):
    """跑一遍连续批处理调度。返回 (records, meta)。

    每个 step 的顺序是固定的，顺序本身就是语义：

        1. 空闲则把时间跳到下一个到达时刻（不能凭空推进）
        2. **准入**：从等待队列头部取请求，同时受槽位与 KV 池限制
        3. 填 prefill token 预算（从已在 prefill 的请求队头开始）
        4. 推进 decode（每个在跑的请求 +1 token）
        5. 结算本步代价、推进时间、收尾

    第 2 步与第 3 步之间不做抢占，抢占只发生在「准入失败」时——
    这和真实调度器的形态一致：先看能不能塞进去，塞不进去才考虑驱逐。

    `closed=(并发数, 总请求数)` 切换成 **closed loop（固定并发）**：
    在途请求数恒不超过并发数，一条完成就立刻补一条（零思考时间）。
    这不需要预先知道到达时刻，因为下一个请求的发出时间依赖于
    上一个请求的返回时间。**open loop 与 closed loop 不能互推**，
    就是因为这一点：closed loop 下等待队列永远为空。
    """
    if closed is not None:
        c_max, n_total = closed
        reqs, pending = [], []
        dispatched = 0
        t = 0.0
    else:
        reqs = [_Req(i, t * 1000.0, in_len, out_len)
                for i, t in enumerate(times_s)]
        pending = list(reqs)
        n_total = len(reqs)
        dispatched = n_total
        t = 0.0

    prefilling = []               # 已准入、prefill 未完成
    decoding = []                 # prefill 完成、正在出 token
    done = []
    n_preempt = 0
    steps = 0

    def kv_used():
        return sum(r.kv_tokens(cfg.block_size) for r in prefilling + decoding)

    while (pending or prefilling or decoding or
           (closed is not None and dispatched < n_total)) and steps < max_steps:
        steps += 1

        # --- 0. closed loop：补足在途请求 ---
        if closed is not None:
            while dispatched < n_total and \
                    (len(pending) + len(prefilling) + len(decoding)) < c_max:
                r = _Req(dispatched, t, in_len, out_len)
                dispatched += 1
                reqs.append(r)
                pending.append(r)

        # --- 1. 空闲则跳到下一个到达 ---
        if not prefilling and not decoding and pending:
            if closed is None and pending[0].arrival_ms > t:
                t = pending[0].arrival_ms

        # --- 2. 准入 ---
        while pending and pending[0].arrival_ms <= t:
            r = pending[0]
            need = cfg.kv_blocks(r.in_len) * cfg.block_size
            if need > cfg.kv_token_capacity:
                # 单条请求的 prompt 就超过整个块池：永远进不去，直接判死
                r.rejected = True
                pending.pop(0)
                continue
            slots = cfg.max_num_seqs - (len(prefilling) + len(decoding))
            if slots <= 0:
                break
            if kv_used() + need > cfg.kv_token_capacity:
                if not cfg.preempt:
                    break
                victim = None
                for c in decoding:
                    if c.preempted:
                        continue
                    if c.kv_tokens(cfg.block_size) >= need:
                        if victim is None or c.kv_tokens(cfg.block_size) > \
                                victim.kv_tokens(cfg.block_size):
                            victim = c
                if victim is None:
                    break
                # 驱逐 = 丢弃已算出的 token，回等待队列头部，重算
                decoding.remove(victim)
                victim.out_done = 0
                victim.in_done = 0
                victim.preempted = True
                victim.ttft_ms = None
                pending.insert(0, victim)
                n_preempt += 1
                continue
            pending.pop(0)
            prefilling.append(r)

        # --- 3. prefill token 预算 ---
        budget = cfg.max_num_batched_tokens
        p_tokens = 0
        p_active = []
        for r in prefilling:
            if budget <= 0:
                break
            take = min(budget, r.in_len - r.in_done)
            if take <= 0:
                continue
            r.chunk = take
            budget -= take
            p_tokens += take
            p_active.append(r)

        # --- 4. 本步代价 ---
        # decode 批次必须取「本步开始时已在 decode 的请求」的快照：
        # 本步刚做完 prefill 的请求已经产出了首 token，**不能再多吃一步 decode**。
        # 不拍这个快照就会少算一步，单请求 E2EL 会差一个 TPOT。
        d_batch = list(decoding)
        n_d = len(d_batch)
        step_ms = cfg.prefill_step_ms(p_tokens) + cfg.decode_step_ms(n_d)
        if step_ms <= 0:
            # 没有任何工作可做：只能是纯等待，跳到下一个到达时刻
            if pending:
                t = max(t, pending[0].arrival_ms)
                continue
            break

        # --- 5. 推进时间并收尾 ---
        t += step_ms

        for r in p_active:
            r.in_done += r.chunk
            r.chunk = 0
            if r.in_done >= r.in_len:
                # prefill 完成即产出首 token
                r.ttft_ms = t - r.arrival_ms
                r.out_done = 1
                prefilling.remove(r)
                if r.out_len <= 1:
                    r.e2el_ms = t - r.arrival_ms
                    r.finished = True
                    done.append(r)
                else:
                    decoding.append(r)

        for r in d_batch:
            r.out_done += 1
            if r.out_done >= r.out_len:
                r.e2el_ms = t - r.arrival_ms
                r.finished = True
                decoding.remove(r)
                done.append(r)

    records = []
    for r in done:
        tpot = None
        if r.out_len > 1 and r.e2el_ms is not None and r.ttft_ms is not None:
            tpot = (r.e2el_ms - r.ttft_ms) / float(r.out_len - 1)
        records.append(dict(
            idx=r.idx, arrival_ms=r.arrival_ms,
            ttft_ms=r.ttft_ms, tpot_ms=tpot, e2el_ms=r.e2el_ms,
            in_len=r.in_len, n_out=r.out_len, preempted=r.preempted,
        ))
    finished_idx = set(r.idx for r in done)
    meta = dict(
        duration_ms=t, steps=steps, n_preempt=n_preempt,
        n_rejected=sum(1 for r in reqs if r.rejected),
        n_finished=len(done),
        total_input_tokens=sum(r.in_len for r in reqs if r.idx in finished_idx),
        total_output_tokens=sum(r.out_len for r in reqs if r.idx in finished_idx),
        n_offered=len(reqs),
    )
    return records, meta

## 第 1 节 · 字段映射：队列模型输出 ↔ vLLM `--save-result`

三组：

- **A 组** —— 队列模型能给，可与压测输出逐字段对照
- **B 组** —— 需要真实流式时序，队列模型**给不出**
- **C 组** —— 服务端才有，客户端压测看不到

In [3]:
# ---- 摘自 tools/profiling_bench/serving_schema.py ----
FIELD_MAP = [
    # ---- 汇总：条数、时长、token 数 ----
    ("completed", "completed", "A", "verified",
     "成功完成的请求数。分母口径要一致：被拒/超时/答案错的算不算，各工具不同。"),
    ("duration", "duration", "A", "verified", "整段挂钟秒数。"),
    ("n_offered", "—", "A", "inferred",
     "发出的请求数。vLLM 结果里没有独立的 offered 字段，只能从 num_prompts 推。"),
    ("total_input_tokens", "total_input_tokens", "A", "verified", "输入 token 总数。"),
    ("total_output_tokens", "total_output_tokens", "A", "verified", "输出 token 总数。"),

    # ---- 吞吐 ----
    ("request_throughput", "request_throughput", "A", "verified", "req/s。"),
    ("output_throughput", "output_throughput", "A", "verified",
     "输出 tok/s。**这是判断 decode 是否饱和的那一个数**。"),
    ("total_token_throughput", "total_token_throughput", "A", "verified",
     "输入+输出 tok/s。分母里混了 prefill，和 output 不是一回事。"),

    # ---- TTFT ----
    ("mean_ttft_ms", "mean_ttft_ms", "A", "verified", "均值。分布式/排队场景下几乎总是骗人的。"),
    ("median_ttft_ms", "median_ttft_ms", "A", "verified", "中位数。"),
    ("p90_ttft_ms", "p90_ttft_ms", "A", "cond",
     "**条件出现**：只有 --metric-percentiles 里包含 90 才有；默认值是 99，"
     "所以默认跑出来的结果文件里没有这个键。"),
    ("p99_ttft_ms", "p99_ttft_ms", "A", "cond",
     "同样是条件出现，只不过 99 是默认值，所以默认就有。"),

    # ---- TPOT ----
    ("mean_tpot_ms", "mean_tpot_ms", "A", "verified", "每条请求算一次再聚合。"),
    ("median_tpot_ms", "median_tpot_ms", "A", "verified", "同上，中位数。"),
    ("p90_tpot_ms", "p90_tpot_ms", "A", "cond", "条件出现，同 p90_ttft_ms。"),
    ("p99_tpot_ms", "p99_tpot_ms", "A", "cond", "条件出现，默认有。"),

    # ---- E2EL ----
    ("mean_e2el_ms", "mean_e2el_ms", "A", "verified", "端到端时延。"),
    ("median_e2el_ms", "median_e2el_ms", "A", "verified", "同上，中位数。"),
    ("p99_e2el_ms", "p99_e2el_ms", "A", "cond", "条件出现，默认有。"),
    ("—", "—", "A", "self",
     "注意 goodput 结构里的 ttft_ms / tpot_ms 是 **SLO 阈值**，不是测量值；"
     "别把它和逐请求真值数组 ttfts 看混。"),

    # ---- goodput ----
    ("slo_attainment", "—", "A", "inferred",
     "达标率。由 --goodput 的 SLO 口径算出，但结果文件里通常只给 goodput 数，"
     "不给达标率本身；要做容量边界判读得自己算。"),
    ("request_goodput", "request_goodput", "A", "inferred",
     "满足 SLO 的 req/s。名字是推断的：文档只说明 --goodput 会加 goodput 指标，"
     "未给字段名。**不要拿它做自动化断言。**"),
    ("output_token_goodput", "—", "A", "inferred",
     "满足 SLO 的输出 tok/s。vLLM 侧口径请自行核对。"),

    # ---- 分位比值（本系列自造，工具里没有）----
    ("ttft_p99_over_p50", "—", "A", "self",
     "自造合成量。工具不报这个比值，但它是**唯一能把「正在变饱和」和"
     "「已经饱和」区分开**的读数。"),

    # ---- ITL：模型给不出 ----
    ("—", "mean_itl_ms", "B", "verified",
     "相邻流式输出的平均间隔。需要**流式**逐块到达时刻；队列模型只有逐请求的"
     "首/末时刻，没有块边界，算不出。"),
    ("—", "median_itl_ms", "B", "verified", "同上。"),
    ("—", "std_itl_ms", "B", "verified", "同上。"),
    ("—", "p99_itl_ms", "B", "versioned", "同上。"),
    ("—", "ttfts", "B", "verified",
     "逐请求 TTFT 原始数组（--save-detailed）。模型能给，但工具侧才是真值。"),
    ("—", "itls", "B", "verified", "逐请求 ITL 数组。模型给不出。"),

    # ---- 服务端才有 ----
    ("n_preemptions_in_model", "—", "C", "self",
     "模型内抢占次数。真实 vLLM 有 vllm:num_preemptions_total 指标，"
     "但**客户端压测结果里没有**——它是服务端 Prometheus 指标。"),
    ("—", "—", "C", "verified",
     "KV 块占用率、等待队列长度、prefix cache 命中率：都是服务端指标。"
     "客户端只能看到它们**造成的结果**（TTFT 起飞的形状），看不到原因。"),
]



GROUP_TITLE = {
    "A": "A 组 · 队列模型能给，可与压测输出逐字段对照",
    "B": "B 组 · 需要真实流式时序，队列模型给不出",
    "C": "C 组 · 服务端才有，客户端压测看不到",
}



CONF_TAG = {
    "verified": "已核验  ",
    "versioned": "版本相关",
    "cond": "条件出现",
    "inferred": "推断    ",
    "self": "自造    ",
}



def print_field_map():
    print("=" * 108)
    print("字段映射：队列模型输出 ↔ vLLM `--save-result`")
    print("=" * 108)
    print("可信度说明：")
    print("  已核验   vLLM 文档或 benchmark_serving.py 源码里直接出现的名字")
    print("  版本相关 分位数字段名 p{N}_<metric>_ms，跨版本可能增删；换版本先跑一次看键名")
    print("  推断     由文档描述推出的名字，未经一手确认，不要用于自动化断言")
    print("  自造     本系列自己的合成量，工具里没有对应字段")
    print()
    for g in ("A", "B", "C"):
        print("-" * 108)
        print(GROUP_TITLE[g])
        print("-" * 108)
        print("  %-40s %-26s %-8s %s" % ("模型字段", "vLLM 字段", "可信度", "说明"))
        for our, vllm, grp, conf, note in FIELD_MAP:
            if grp != g:
                continue
            print("  %-40s %-26s %-8s %s" % (our, vllm, CONF_TAG[conf], note))
        print()
    print("关键提醒：**不要把 B/C 组的量当成「也是模型算的」**。")
    print("队列模型回答「拐点在哪、由什么决定」；它不回答「ITL 的尾部形态」。")
    print()



print_field_map()

字段映射：队列模型输出 ↔ vLLM `--save-result`
可信度说明：
  已核验   vLLM 文档或 benchmark_serving.py 源码里直接出现的名字
  版本相关 分位数字段名 p{N}_<metric>_ms，跨版本可能增删；换版本先跑一次看键名
  推断     由文档描述推出的名字，未经一手确认，不要用于自动化断言
  自造     本系列自己的合成量，工具里没有对应字段

------------------------------------------------------------------------------------------------------------
A 组 · 队列模型能给，可与压测输出逐字段对照
------------------------------------------------------------------------------------------------------------
  模型字段                                     vLLM 字段                    可信度      说明
  completed                                completed                  已核验      成功完成的请求数。分母口径要一致：被拒/超时/答案错的算不算，各工具不同。
  duration                                 duration                   已核验      整段挂钟秒数。
  n_offered                                —                          推断       发出的请求数。vLLM 结果里没有独立的 offered 字段，只能从 num_prompts 推。
  total_input_tokens                       total_input_tokens         已核验      输入 token 总数。
  total_output_tokens                    

> **不要把 B/C 组的量当成「也是模型算的」。** 队列模型回答「拐点在哪、由什么决定」；它**不回答「ITL 的尾部形态」**。

## 第 2 节 · ITL 与 TPOT 的分母：同一个流，两个数可以差几倍

**这是本文最容易读错的一处。** vLLM 官方文档给的定义：

- **ITL** 记录**相邻两次流式输出**之间的间隔。一个流式输出里如果含多个 token（投机解码下一个 engine step 接受多个 draft token 时就会这样），**这些 token 不额外产生 ITL 样本**。
- **TPOT** 是**每条请求算一次**、排除首 token 后摊到每个输出 token 上，再跨请求聚合。

于是两者相差一个分母：

$$\frac{\text{mean\_ITL}}{\text{TPOT}} = \frac{n_{\text{out}} - 1}{n_{\text{streamed\_gaps}}}$$

In [4]:
# ---- 摘自 tools/profiling_bench/serving_schema.py ----
def itl_vs_tpot(chunks, ttft_ms=100.0, e2el_ms=180.0):
    """`chunks` = 每个流式输出携带的 token 数序列。

    返回 (n_out, n_gaps, mean_itl, tpot, ratio)。
    ITL 只在**块边界**上产生样本，块内多个 token 不产生样本。
    """
    n_out = sum(chunks)
    n_gaps = len(chunks) - 1
    if n_gaps <= 0:
        mean_itl = float("nan")
    else:
        mean_itl = (e2el_ms - ttft_ms) / n_gaps
    tpot = (e2el_ms - ttft_ms) / (n_out - 1) if n_out > 1 else float("nan")
    ratio = (mean_itl / tpot) if tpot and tpot > 0 else float("nan")
    return n_out, n_gaps, mean_itl, tpot, ratio



def print_itl_demo():
    print("=" * 96)
    print("ITL 与 TPOT 的分母：同一个流，两个数可以差几倍")
    print("=" * 96)
    print("固定 TTFT = 100 ms、E2EL = 180 ms（即解码段共 80 ms），只改每个流式输出带几个 token。")
    print()
    print("%-22s %6s %7s %12s %12s %10s" %
          ("每个流式输出带 token 数", "n_out", "n_gaps", "mean ITL", "TPOT", "ITL/TPOT"))
    print("-" * 96)
    cases = [
        ("全部 1 个（常规解码）", [1, 1, 1, 1, 1]),
        ("2 个 / 1 个 / 2 个", [2, 1, 2]),
        ("2 个 / 2 个 / 2 个", [2, 2, 2]),
        ("3 个 / 3 个", [3, 3]),
    ]
    for label, chunks in cases:
        n_out, n_gaps, itl, tpot, ratio = itl_vs_tpot(chunks)
        print("%-22s %6d %7d %10.1f ms %10.1f ms %10.2f" %
              (label, n_out, n_gaps, itl, tpot, ratio))
    print("-" * 96)
    print("闭式关系：mean_ITL / TPOT = (n_out − 1) / n_streamed_gaps")
    print()
    print("第二行（2 个 / 1 个 / 2 个）就是 vLLM 文档里的例子：两次 40 ms 的流式间隔 →")
    print("mean ITL = 40 ms；同一份时序的 TPOT = (180 − 100) / (5 − 1) = 20 ms/token。")
    print("即 mean ITL = 2 × TPOT。**两个数都对，因为问的不是同一件事。**")
    print()
    print("第一行（每个输出恰好一个 token）是两者相近的**唯一**情形，")
    print("比值 1.00——「ITL 和 TPOT 差不多」这句话只在这种情况下成立。")
    print()
    print("实操含义：开投机解码后每个流式输出接受的 token 数上升，")
    print("n_streamed_gaps 下降，mean ITL 按 (n_out−1)/n_gaps 同比例放大，")
    print("而 TPOT（每条请求摊到每个 token）不受这个口径影响。")
    print("只盯 ITL 会把这个口径放大读成「服务变慢了」。")
    print()



print_itl_demo()

ITL 与 TPOT 的分母：同一个流，两个数可以差几倍
固定 TTFT = 100 ms、E2EL = 180 ms（即解码段共 80 ms），只改每个流式输出带几个 token。

每个流式输出带 token 数         n_out  n_gaps     mean ITL         TPOT   ITL/TPOT
------------------------------------------------------------------------------------------------
全部 1 个（常规解码）                5       4       20.0 ms       20.0 ms       1.00
2 个 / 1 个 / 2 个             5       2       40.0 ms       20.0 ms       2.00
2 个 / 2 个 / 2 个             6       2       40.0 ms       16.0 ms       2.50
3 个 / 3 个                   6       1       80.0 ms       16.0 ms       5.00
------------------------------------------------------------------------------------------------
闭式关系：mean_ITL / TPOT = (n_out − 1) / n_streamed_gaps

第二行（2 个 / 1 个 / 2 个）就是 vLLM 文档里的例子：两次 40 ms 的流式间隔 →
mean ITL = 40 ms；同一份时序的 TPOT = (180 − 100) / (5 − 1) = 20 ms/token。
即 mean ITL = 2 × TPOT。**两个数都对，因为问的不是同一件事。**

第一行（每个输出恰好一个 token）是两者相近的**唯一**情形，
比值 1.00——「ITL 和 TPOT 差不多」这句话只在这种情况下成立。

实操含义：开投机解码后每个流式输出接受的 token 数上升，
n_st

> **实操含义**：开投机解码后每个流式输出接受的 token 数上升，$n_{\text{streamed\_gaps}}$ 下降，**mean ITL 按 $(n_{\text{out}}-1)/n_{\text{gaps}}$ 同比例放大**，而 TPOT（每条请求摊到每个 token）不受这个口径影响。
>
> **只盯 ITL 会把这个口径放大读成「服务变慢了」。**

## 第 3 节 · 落盘与逐字段回读

**这步在做什么**：跑一遍队列模型 → 按 `--save-result` 的形状组织成 JSON → 回读，断言 A 组要求的字段一个不缺。

In [5]:
# ---- 摘自 tools/profiling_bench/serving_schema.py ----
def build_result(summary, cfg, slo_ttft_ms, slo_tpot_ms, label="serving_model"):
    """按 vLLM `--save-result` 的形状组织一份结果。"""
    out = {
        "label": label,
        "args": {
            "max_num_seqs": cfg.max_num_seqs,
            "max_num_batched_tokens": cfg.max_num_batched_tokens,
            "block_size": cfg.block_size,
            "kv_token_capacity": cfg.kv_token_capacity,
        },
        "completed": summary["completed"],
        "duration": summary["duration"],
        "total_input_tokens": summary["total_input_tokens"],
        "total_output_tokens": summary["total_output_tokens"],
        "request_throughput": summary["request_throughput"],
        "output_throughput": summary["output_throughput"],
        "total_token_throughput": summary["total_token_throughput"],
        "mean_ttft_ms": summary["mean_ttft_ms"],
        "median_ttft_ms": summary["median_ttft_ms"],
        "p90_ttft_ms": summary["p90_ttft_ms"],
        "p99_ttft_ms": summary["p99_ttft_ms"],
        "mean_tpot_ms": summary["mean_tpot_ms"],
        "median_tpot_ms": summary["median_tpot_ms"],
        "p90_tpot_ms": summary["p90_tpot_ms"],
        "p99_tpot_ms": summary["p99_tpot_ms"],
        "mean_e2el_ms": summary["mean_e2el_ms"],
        "median_e2el_ms": summary["median_e2el_ms"],
        "p99_e2el_ms": summary["p99_e2el_ms"],
    }
    if slo_ttft_ms is not None and slo_tpot_ms is not None:
        out["goodput"] = {
            "ttft_ms": slo_ttft_ms,
            "tpot_ms": slo_tpot_ms,
            "request_goodput": summary.get("request_goodput"),
            "output_token_goodput": summary.get("output_token_goodput"),
            "slo_attainment": summary.get("slo_attainment"),
        }
    return out



def check(explicit=None):
    """跑一遍、落盘、回读，断言 A 组字段一个不缺。"""
    cfg = ServerCfg(max_num_seqs=32, max_num_batched_tokens=2048,
                    kv_token_capacity=262144)
    times, _ = arrival_times("poisson", 8.0, 300, seed=0)
    recs, meta = simulate(cfg, times, 1024, 128)
    s = summarize(recs, meta, cfg, 300.0, 30.0)
    result = build_result(s, cfg, 300.0, 30.0)

    present = set(result.keys()) | set(result.get("goodput", {}).keys())
    required = [v for _, v, g, _, _ in FIELD_MAP if g == "A" and v != "—"]
    our_keys = set(k for k, v, g, _, _ in FIELD_MAP
                   if g == "A" and k != "—" and v == "—")
    missing = [v for v in required if v not in present]
    unknown = [k for k in present
               if k not in required and k not in our_keys
               and k not in ("label", "args", "goodput",
                             "ttft_ms", "tpot_ms")]

    print("=" * 88)
    print("逐字段回读校验")
    print("=" * 88)
    print("A 组要求字段 %d 个，实际落盘 %d 个。" % (len(required), len(present)))
    if missing:
        print("**缺失**：%s" % ", ".join(missing))
    else:
        print("缺失：无")
    if unknown:
        print("额外字段（不在映射表里）：%s" % ", ".join(sorted(unknown)))
    print()
    if explicit:
        with open(explicit, "w") as fh:
            json.dump(result, fh, indent=2, ensure_ascii=False)
        print("已写入 %s" % explicit)
        print()
    print("落盘样例（前 14 行）：")
    for i, line in enumerate(json.dumps(result, indent=2,
                                        ensure_ascii=False).splitlines()):
        if i >= 14:
            break
        print("  " + line)
    print()
    return not missing



ok = check()
print('\nA 组字段校验: %s' % ('通过' if ok else '失败'))

逐字段回读校验
A 组要求字段 19 个，实际落盘 26 个。
缺失：无

落盘样例（前 14 行）：
  {
    "label": "serving_model",
    "args": {
      "max_num_seqs": 32,
      "max_num_batched_tokens": 2048,
      "block_size": 16,
      "kv_token_capacity": 262144
    },
    "completed": 300,
    "duration": 40.12513788564671,
    "total_input_tokens": 307200,
    "total_output_tokens": 38400,
    "request_throughput": 7.476609821378681,
    "output_throughput": 957.0060571364712,


A 组字段校验: 通过


> **一个容易看错的地方**：goodput 结构里的 `ttft_ms` / `tpot_ms` 是 **SLO 阈值**，不是测量值。别把它和逐请求真值数组 `ttfts` 看混。

## 小结

- **同名不同义是常态**，跨工具比数之前必须逐字段对齐，并对齐到「可信度」这一级。
- **模型给不出的量要写出来**（B 组：ITL 族；C 组：服务端指标）。不写就会有人以为队列模型能替代压测。
- **ITL 与 TPOT 差一个分母**，比值 = $(n_{\text{out}}-1)/n_{\text{streamed\_gaps}}$。投机解码会让 mean ITL 上升而每请求生成速度不变。
- **推断项不要用于自动化断言**——`request_goodput` 的字段名就是推断的。

## 参考文献

- [vLLM 文档](https://docs.vllm.ai/) —— `--save-result` / `--save-detailed` / `--metric-percentiles` / `--goodput` 的字段定义
- vLLM `benchmarks/benchmark_serving.py`